# 05 · Adversarial Retraining & Final Results (Stage 5 — Experiment E4)
Implements **Section 8.6**, experiment **E4** and the **Section 13** deliverables:

1. Augment the original training set with **20 %** and **50 %** of `variant_train_pool` (nested subsets: the 20 % sample is contained in the 50 % sample; stratified by attack class).
2. Retrain all 4 models × 3 seeds with **identical** hyperparameters (imported from `ids_common.py`; the class-weight setting is read from `config/training_config.json`).
3. Evaluate each model on (a) the **original test set** (performance cost, false alarms) and (b) `variant_test_heldout` (**Recovery**). The scripted control set (E3) is also evaluated when present.
4. Produce the final publication-ready tables and the 3 main figures in `results/`.

The share-0 % row is the E1/E2 baseline: the saved baseline models are re-evaluated here with the same code, so every number in the final tables comes from one evaluation path.

**Validation set:** early stopping still uses the **original** validation set (no variants), so differences come only from the augmented training data.

**Runtime:** T4 GPU recommended; 24 training runs take about 40–80 min. **Resumable**: finished runs are saved to Drive and skipped on re-run.

**Prerequisites:** notebooks 01, 02 and 04 have been run.

## 1 · Mount Drive, configuration, shared module

In [ ]:
import os, sys, json, time, gc
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/IDS_Polymorphic_Project'
else:
    BASE_DIR = os.path.abspath('./IDS_Polymorphic_Project')

PCFG = json.load(open(os.path.join(BASE_DIR, 'config', 'project_config.json')))
FCFG = json.load(open(os.path.join(BASE_DIR, 'config', 'feature_config.json')))
TCFG = json.load(open(os.path.join(BASE_DIR, 'config', 'training_config.json')))
DIRS = {k: v.replace(PCFG['dirs']['base'], BASE_DIR) for k, v in PCFG['dirs'].items()}
DIRS['code'] = os.path.join(BASE_DIR, 'code')
DIRS['baseline_models'] = os.path.join(DIRS['models'], 'baseline')
DIRS['runs05'] = os.path.join(DIRS['results'], '05_runs')
DIRS['figures'] = os.path.join(DIRS['results'], 'figures')
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, DIRS['code'])
import importlib, ids_common
importlib.reload(ids_common)
from ids_common import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, torch

CLASS_NAMES = PCFG['class_names']; BENIGN_ID = PCFG['benign_id']; ATTACK_CLASSES = PCFG['attack_classes']
LABEL_TO_ID = PCFG['label_to_id']; MODEL_SEEDS = PCFG['model_seeds']; DATA_SEED = PCFG['data_seed']
FEATURES = FCFG['feature_names']; N_CLASSES = len(CLASS_NAMES)
DEVICE = get_device()
CLASS_WEIGHTS = {int(k): v for k, v in PCFG['class_weights_balanced'].items()} if TCFG['use_class_weights'] else None
print('device:', DEVICE, '| class weights used:', TCFG['use_class_weights'])
print('RF:', RF_HPARAMS, '| DL:', DL_HPARAMS)

## 2 · Experiment settings and data

In [ ]:
SHARES = [0.20, 0.50]          # shares of variant_train_pool added to the training set
FORCE_RETRAIN = False
VERBOSE_EPOCHS = False

P = DIRS['processed']
X_train = np.load(os.path.join(P, 'X_train.npy')); y_train = np.load(os.path.join(P, 'y_train.npy'))
X_val   = np.load(os.path.join(P, 'X_val.npy'));   y_val   = np.load(os.path.join(P, 'y_val.npy'))
X_test  = np.load(os.path.join(P, 'X_test.npy'));  y_test  = np.load(os.path.join(P, 'y_test.npy'))
scaler  = joblib.load(os.path.join(P, 'scaler.joblib'))

pool = pd.read_parquet(os.path.join(DIRS['variants'], 'variant_train_pool.parquet'))
held = pd.read_parquet(os.path.join(DIRS['variants'], 'variant_test_heldout.parquet'))
scr_path = os.path.join(DIRS['variants'], 'scripted_control.parquet')
scr = pd.read_parquet(scr_path) if os.path.exists(scr_path) else pd.DataFrame()
if len(pool) == 0 or len(held) == 0:
    raise ValueError('variant_train_pool / variant_test_heldout are empty: run notebook 04 with validated LLM variants first.')
if 'variant_id' in pool and 'variant_id' in held:
    assert set(pool['variant_id']).isdisjoint(held['variant_id']), 'pool and held-out overlap!'

def scale(frame):
    return scaler.transform(frame[FEATURES].to_numpy(dtype=np.float32)).astype(np.float32)

X_pool, y_pool = scale(pool), pool['label_id'].to_numpy()
EVAL_ATTACK_SETS = {'heldout_llm': (scale(held), held['label_id'].to_numpy())}
if len(scr):
    EVAL_ATTACK_SETS['scripted'] = (scale(scr), scr['label_id'].to_numpy())
att_mask = y_test != BENIGN_ID
EVAL_ATTACK_SETS['original_attacks'] = (X_test[att_mask], y_test[att_mask])

print(f'train {X_train.shape} | val {X_val.shape} | test {X_test.shape}')
print('variant_train_pool per class  :', pool['Label'].value_counts().reindex(ATTACK_CLASSES).fillna(0).astype(int).to_dict())
print('variant_test_heldout per class:', held['Label'].value_counts().reindex(ATTACK_CLASSES).fillna(0).astype(int).to_dict())
print('scripted control rows         :', len(scr))

## 3 · Build the augmented training sets (nested, stratified by class)
Each class of the pool is shuffled once with `DATA_SEED`; the first 20 % / 50 % of every class are taken. The same variant subsets are used for all models and seeds, so only model randomness differs between seeds. The selected `variant_id`s are saved for reproducibility.

In [ ]:
rng = np.random.default_rng(DATA_SEED)
order = {c: rng.permutation(np.where(pool['Label'].to_numpy() == c)[0]) for c in ATTACK_CLASSES}
AUG = {}
selection_log = {}
for share in SHARES:
    idx = np.concatenate([o[:int(round(share * len(o)))] for o in order.values()]).astype(int)
    X_aug = np.concatenate([X_train, X_pool[idx]]).astype(np.float32)
    y_aug = np.concatenate([y_train, y_pool[idx]])
    perm = np.random.default_rng(DATA_SEED).permutation(len(y_aug))
    AUG[share] = (X_aug[perm], y_aug[perm])
    selection_log[f'{int(share * 100)}pct'] = {
        'n_variants_added': int(len(idx)),
        'per_class': {c: int((y_pool[idx] == LABEL_TO_ID[c]).sum()) for c in ATTACK_CLASSES},
        'variant_ids': pool['variant_id'].to_numpy()[idx].tolist() if 'variant_id' in pool else idx.tolist()}
    print(f'share {share:.0%}: +{len(idx):,} variants -> train size {len(y_aug):,} '
          f'| added per class {selection_log[f"{int(share * 100)}pct"]["per_class"]}')
if len(SHARES) > 1:
    a, b = sorted(SHARES)[:2]
    assert set(selection_log[f'{int(a * 100)}pct']['variant_ids']) <= set(selection_log[f'{int(b * 100)}pct']['variant_ids'])
with open(os.path.join(DIRS['results'], '05_augmentation_selection.json'), 'w') as fh:
    json.dump(selection_log, fh)

## 4 · E4 — retrain and evaluate (share 0 % = baseline models re-evaluated)
For each (share, model, seed) we store:
- full multi-class metrics on the **original test set** (`compute_metrics`), and
- attack-set metrics (recall per class, evasion) on **held-out LLM variants**, the **scripted control** set and the **original test attacks** (`attack_set_metrics`).

In [ ]:
def macro_attack_recall(m):
    vals = [m[f'recall_{c}'] for c in ATTACK_CLASSES if m.get(f'n_{c}', 0) > 0]
    return float(np.mean(vals)) if vals else float('nan')

def evaluate_model(name, model):
    out = {}
    pred_test = predict(name, model, X_test, device=DEVICE)
    m_test, cm_test = compute_metrics(y_test, pred_test, CLASS_NAMES, BENIGN_ID)
    out['test'] = m_test; out['test_cm'] = cm_test.tolist()
    for set_name, (X, y) in EVAL_ATTACK_SETS.items():
        m = attack_set_metrics(y, predict(name, model, X, device=DEVICE), CLASS_NAMES, BENIGN_ID)
        m['recall_macro_attack'] = macro_attack_recall(m)
        out[set_name] = m
    return out

def run_file(share, name, seed):
    return os.path.join(DIRS['runs05'], f'share{int(share * 100):03d}_{name}_seed{seed}.json')

all_results = []
t0 = time.time()
for share in [0.0] + SHARES:
    model_dir = DIRS['baseline_models'] if share == 0 else os.path.join(DIRS['models'], f'retrained_{int(share * 100)}pct')
    os.makedirs(model_dir, exist_ok=True)
    for seed in MODEL_SEEDS:
        for name in MODEL_NAMES:
            rf = run_file(share, name, seed)
            mpath = model_path(model_dir, name, seed)
            if not FORCE_RETRAIN and os.path.exists(rf) and (share == 0 or os.path.exists(mpath)):
                all_results.append(json.load(open(rf))); print(f'[skip] share {share:.0%} {name} seed {seed}'); continue

            info = {}
            if share == 0:
                if not os.path.exists(mpath):
                    raise FileNotFoundError(f'baseline model missing: {mpath} (run notebook 02)')
                model = load_model(name, mpath, device=DEVICE)
            else:
                Xa, ya = AUG[share]
                print(f'\n=== retrain {name} | share {share:.0%} | seed {seed} | n={len(ya):,} ===')
                if name == 'RandomForest':
                    model, info = train_rf(Xa, ya, seed)
                else:
                    model, info = train_dl(name, Xa, ya, X_val, y_val, seed, N_CLASSES, device=DEVICE,
                                           class_weights=CLASS_WEIGHTS, verbose=VERBOSE_EPOCHS)
                save_model(name, model, mpath, len(FEATURES), N_CLASSES)

            res = evaluate_model(name, model)
            res.update({'share': share, 'model': name, 'seed': seed,
                        'train_time_s': info.get('train_time_s'), 'best_epoch': info.get('best_epoch'),
                        'epochs_run': info.get('epochs_run')})
            with open(rf, 'w') as fh:
                json.dump(res, fh)
            all_results.append(res)
            print(f'  share {share:.0%} {name} seed {seed}: test F1-macro {res["test"]["f1_macro"]:.4f} | '
                  f'FPR {res["test"]["fpr"]:.4f} | held-out recall {res["heldout_llm"]["recall_macro_attack"]:.4f} | '
                  f'held-out evasion {res["heldout_llm"]["evasion_rate"]:.4f}')
            del model; gc.collect()
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
print(f'\n{len(all_results)} runs ready in {(time.time() - t0) / 60:.1f} min')

## 5 · Flatten results, Recovery and cost on the original data

In [ ]:
rows = []
for r in all_results:
    row = {'share': r['share'], 'model': r['model'], 'seed': r['seed'], 'train_time_s': r.get('train_time_s')}
    for k in ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'fpr', 'detection_rate', 'evasion_rate']:
        row[f'test_{k}'] = r['test'][k]
    for c in CLASS_NAMES:
        row[f'test_recall_{c}'] = r['test'][f'recall_{c}']; row[f'test_f1_{c}'] = r['test'][f'f1_{c}']
    for s in [k for k in EVAL_ATTACK_SETS]:
        row[f'{s}_recall'] = r[s]['recall_macro_attack']
        row[f'{s}_evasion'] = r[s]['evasion_rate']
        for c in ATTACK_CLASSES:
            row[f'{s}_recall_{c}'] = r[s].get(f'recall_{c}', np.nan)
            row[f'{s}_evasion_{c}'] = r[s].get(f'evasion_{c}', np.nan)
    rows.append(row)
R = pd.DataFrame(rows)
R['model'] = pd.Categorical(R['model'], MODEL_NAMES, ordered=True)
R = R.sort_values(['share', 'model', 'seed']).reset_index(drop=True)

# paired (same model, same seed) deltas vs the baseline
base = R[R['share'] == 0].set_index(['model', 'seed'])
delta_cols = ['heldout_llm_recall', 'heldout_llm_evasion', 'test_f1_macro', 'test_fpr', 'test_accuracy',
              'original_attacks_recall'] + [f'heldout_llm_recall_{c}' for c in ATTACK_CLASSES]
if 'scripted_recall' in R:
    delta_cols.append('scripted_recall')
for c in delta_cols:
    R[f'delta_{c}'] = [r[c] - base.loc[(r['model'], r['seed']), c] for _, r in R.iterrows()]
R = R.rename(columns={'delta_heldout_llm_recall': 'recovery'})
R.to_csv(os.path.join(DIRS['results'], '05_E4_runs_all_seeds.csv'), index=False)
print(R[['share', 'model', 'seed', 'test_f1_macro', 'test_fpr', 'heldout_llm_recall', 'heldout_llm_evasion', 'recovery']]
      .round(4).to_string(index=False))

In [ ]:
SUMMARY_COLS = ['test_accuracy', 'test_f1_macro', 'test_fpr', 'original_attacks_recall',
                'heldout_llm_recall', 'heldout_llm_evasion', 'recovery', 'delta_test_f1_macro', 'delta_test_fpr']
if 'scripted_recall' in R:
    SUMMARY_COLS += ['scripted_recall']
num_e4, pretty_e4 = mean_std_table(R, ['share', 'model'], SUMMARY_COLS)
num_e4.to_csv(os.path.join(DIRS['results'], '05_E4_summary_mean_std.csv'))
pretty_e4.to_csv(os.path.join(DIRS['results'], '05_E4_summary_pretty.csv'))
print('E4 summary (mean ± std over seeds):')
print(pretty_e4[['test_f1_macro', 'test_fpr', 'heldout_llm_recall', 'heldout_llm_evasion', 'recovery']].to_string())

rec_rows = []
for c in ATTACK_CLASSES:
    g = R.groupby(['share', 'model'], observed=True)[[f'heldout_llm_recall_{c}', f'delta_heldout_llm_recall_{c}', f'heldout_llm_evasion_{c}']]
    mu, sd = g.mean(), g.std(ddof=1).fillna(0)
    for (sh, m) in mu.index:
        rec_rows.append({'share': sh, 'model': m, 'class': c,
                         'heldout_recall_mean': mu.loc[(sh, m), f'heldout_llm_recall_{c}'],
                         'recovery_mean': mu.loc[(sh, m), f'delta_heldout_llm_recall_{c}'],
                         'recovery_std': sd.loc[(sh, m), f'delta_heldout_llm_recall_{c}'],
                         'evasion_mean': mu.loc[(sh, m), f'heldout_llm_evasion_{c}']})
rec_cls = pd.DataFrame(rec_rows)
rec_cls.to_csv(os.path.join(DIRS['results'], '05_E4_recovery_per_class.csv'), index=False)
print('\nRecovery per attack class (held-out LLM variants):')
print(rec_cls[rec_cls['share'] > 0].round(4).to_string(index=False))

## 6 · Final publication-ready table (E1 → E2 → E3 → E4)
One row per model, every cell **mean ± std over the 3 seeds**:

| Column | Experiment | Meaning |
|---|---|---|
| E1 F1-macro / FPR | E1 | original test set, baseline models |
| E1 attack recall | E1 | class-balanced recall on original test attacks |
| E2 recall / drop / evasion | E2 | held-out LLM variants, baseline models |
| E3 drop / evasion | E3 | scripted control variants, baseline models |
| E4 recovery / ΔF1 / ΔFPR @ 20 % and 50 % | E4 | retrained models: recovery on held-out variants, change on the original test set |

In [ ]:
def pm(mu, sd, d=3):
    return f'{mu:.{d}f} ± {sd:.{d}f}'

def stat(df, col):
    return df[col].mean(), (df[col].std(ddof=1) if len(df) > 1 else 0.0)

final_rows = []
for name in MODEL_NAMES:
    b = R[(R['share'] == 0) & (R['model'] == name)]
    if b.empty:
        continue
    row = {'Model': name,
           'E1 F1-macro': pm(*stat(b, 'test_f1_macro')),
           'E1 FPR': pm(*stat(b, 'test_fpr'), d=4),
           'E1 attack recall': pm(*stat(b, 'original_attacks_recall'))}
    drop = b['original_attacks_recall'] - b['heldout_llm_recall']
    row['E2 recall'] = pm(*stat(b, 'heldout_llm_recall'))
    row['E2 recall drop'] = pm(drop.mean(), drop.std(ddof=1) if len(drop) > 1 else 0)
    row['E2 evasion'] = pm(*stat(b, 'heldout_llm_evasion'))
    if 'scripted_recall' in b:
        d3 = b['original_attacks_recall'] - b['scripted_recall']
        row['E3 recall drop'] = pm(d3.mean(), d3.std(ddof=1) if len(d3) > 1 else 0)
        row['E3 evasion'] = pm(*stat(b, 'scripted_evasion'))
    for sh in SHARES:
        a = R[(R['share'] == sh) & (R['model'] == name)]
        if a.empty:
            continue
        tag = f'{int(sh * 100)}%'
        row[f'E4@{tag} held-out recall'] = pm(*stat(a, 'heldout_llm_recall'))
        row[f'E4@{tag} recovery'] = pm(*stat(a, 'recovery'))
        row[f'E4@{tag} ΔF1 (orig)'] = pm(*stat(a, 'delta_test_f1_macro'), d=4)
        row[f'E4@{tag} ΔFPR (orig)'] = pm(*stat(a, 'delta_test_fpr'), d=4)
    final_rows.append(row)
FINAL = pd.DataFrame(final_rows).set_index('Model')
FINAL.to_csv(os.path.join(DIRS['results'], 'FINAL_summary_table.csv'))
try:
    md_table = FINAL.T.to_markdown()
except Exception:
    md_table = FINAL.T.to_string()
with open(os.path.join(DIRS['results'], 'FINAL_summary_table.md'), 'w') as fh:
    fh.write('# Final results (mean ± std over seeds ' + ', '.join(map(str, MODEL_SEEDS)) + ')\n\n' + md_table + '\n')
try:
    FINAL.T.to_latex(os.path.join(DIRS['results'], 'FINAL_summary_table.tex'))
except Exception as e:
    print('LaTeX export skipped:', e)
print(FINAL.T.to_string())

## 7 · The 3 main figures (Section 13)
Saved as PNG (300 dpi) and PDF in `results/figures/`:
1. **Fig 1** — performance on original data vs variants, per model (class-balanced attack recall, plus macro-F1 on the original test set).
2. **Fig 2** — evasion rate per attack type and model (held-out LLM variants, and scripted control if present).
3. **Fig 3** — before vs after adversarial retraining (held-out recall, original-test F1 and FPR at 0 / 20 / 50 %).

In [ ]:
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
MCOL = {'RandomForest': '#4a4a4a', 'MLP': '#2f6db5', 'CNN1D': '#d1495b', 'LSTM': '#00798c'}
def save(fig, stem):
    for ext in ('png', 'pdf'):
        fig.savefig(os.path.join(DIRS['figures'], f'{stem}.{ext}'), dpi=300, bbox_inches='tight')

B0 = R[R['share'] == 0]
models_present = [m for m in MODEL_NAMES if m in set(B0['model'])]

# ---------- Fig 1
sets = [('original_attacks_recall', 'Original test attacks', '#4a4a4a'),
        ('heldout_llm_recall', 'Held-out LLM variants (E2)', '#2f6db5')]
if 'scripted_recall' in B0:
    sets.append(('scripted_recall', 'Scripted noise (E3)', '#d1495b'))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={'width_ratios': [2.2, 1]})
x = np.arange(len(models_present)); w = 0.8 / len(sets)
for k, (col, lab, c) in enumerate(sets):
    g = B0.groupby('model', observed=True)[col]
    mu = g.mean().reindex(models_present); sd = g.std(ddof=1).reindex(models_present).fillna(0)
    bars = axes[0].bar(x + k * w - 0.4 + w / 2, mu, w, yerr=sd, capsize=3, color=c, label=lab)
    for b_, v in zip(bars, mu):
        axes[0].text(b_.get_x() + b_.get_width() / 2, v + 0.01, f'{v:.2f}', ha='center', va='bottom', fontsize=7)
axes[0].set_xticks(x); axes[0].set_xticklabels(models_present); axes[0].set_ylim(0, 1.12)
axes[0].set_ylabel('class-balanced attack recall'); axes[0].set_title('Detection on original attacks vs polymorphic variants')
axes[0].legend(frameon=False, fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=3)
g = B0.groupby('model', observed=True)['test_f1_macro']
axes[1].bar(models_present, g.mean().reindex(models_present), yerr=g.std(ddof=1).reindex(models_present).fillna(0),
            capsize=3, color=[MCOL[m] for m in models_present])
lo = max(0, g.mean().min() - 0.1)
axes[1].set_ylim(lo, 1.0); axes[1].set_title('Macro-F1 on original test set (E1)'); axes[1].tick_params(axis='x', rotation=20)
fig.suptitle('Fig. 1 — Baseline performance: original data vs variants (mean ± std, 3 seeds)', y=1.02)
plt.tight_layout(); save(fig, 'fig1_original_vs_variants'); plt.show()

# ---------- Fig 2
ev_sets = [('heldout_llm', 'Held-out LLM variants (E2)')] + ([('scripted', 'Scripted noise (E3)')] if 'scripted_recall' in B0 else [])
fig, axes = plt.subplots(1, len(ev_sets), figsize=(6.5 * len(ev_sets), 4.4), squeeze=False)
xc = np.arange(len(ATTACK_CLASSES)); w = 0.8 / len(models_present)
for ax, (s, title) in zip(axes[0], ev_sets):
    for k, m in enumerate(models_present):
        sub = B0[B0['model'] == m]
        mu = [sub[f'{s}_evasion_{c}'].mean() for c in ATTACK_CLASSES]
        sd = [sub[f'{s}_evasion_{c}'].std(ddof=1) if len(sub) > 1 else 0 for c in ATTACK_CLASSES]
        ax.bar(xc + k * w - 0.4 + w / 2, mu, w, yerr=np.nan_to_num(sd), capsize=3, color=MCOL[m], label=m)
    ax.set_xticks(xc); ax.set_xticklabels(ATTACK_CLASSES); ax.set_ylabel('evasion rate (predicted Benign)')
    ax.set_title(title); ax.set_ylim(0, max(0.05, float(np.nanmax(B0[[f'{s}_evasion_{c}' for c in ATTACK_CLASSES]].to_numpy())) * 1.25))
axes[0, 0].legend(frameon=False, fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=len(models_present))
fig.suptitle('Fig. 2 — Evasion rate per attack type and model (baseline models, mean ± std)', y=1.02)
plt.tight_layout(); save(fig, 'fig2_evasion_rate'); plt.show()

# ---------- Fig 3
shares_all = sorted(R['share'].unique())
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, (col, lab) in zip(axes, [('heldout_llm_recall', 'held-out variant recall (class-balanced)'),
                                 ('test_f1_macro', 'macro-F1 on original test set'),
                                 ('test_fpr', 'FPR on original test set')]):
    for m in models_present:
        g = R[R['model'] == m].groupby('share')[col]
        mu = g.mean().reindex(shares_all); sd = g.std(ddof=1).reindex(shares_all).fillna(0)
        ax.errorbar([s * 100 for s in shares_all], mu, yerr=sd, marker='o', capsize=3, lw=1.8, color=MCOL[m], label=m)
    ax.set_xticks([s * 100 for s in shares_all]); ax.set_xlabel('% of variant pool added to training')
    ax.set_ylabel(lab); ax.set_title(lab)
axes[0].legend(frameon=False, fontsize=8, loc='best')
fig.suptitle('Fig. 3 — Before (0 %) vs after adversarial retraining (mean ± std, 3 seeds)', y=1.02)
plt.tight_layout(); save(fig, 'fig3_before_after_retraining'); plt.show()

In [ ]:
# supplementary: per-class recovery heatmap
sub = rec_cls[rec_cls['share'] > 0]
fig, axes = plt.subplots(1, len(SHARES), figsize=(5.5 * len(SHARES), 3.8), squeeze=False)
vmax = max(0.05, float(np.nanmax(np.abs(sub['recovery_mean'].to_numpy())))) if len(sub) else 0.05
for ax, sh in zip(axes[0], SHARES):
    piv = sub[sub['share'] == sh].pivot(index='model', columns='class', values='recovery_mean') \
        .reindex(index=models_present, columns=ATTACK_CLASSES)
    im = ax.imshow(piv.to_numpy(dtype=float), cmap='RdBu', vmin=-vmax, vmax=vmax)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.iat[i, j]
            ax.text(j, i, 'n/a' if pd.isna(v) else f'{v:+.3f}', ha='center', va='center', fontsize=9)
    ax.set_xticks(range(len(ATTACK_CLASSES))); ax.set_xticklabels(ATTACK_CLASSES, rotation=20)
    ax.set_yticks(range(len(models_present))); ax.set_yticklabels(models_present)
    ax.set_title(f'Recovery @ {int(sh * 100)} % (Δ held-out recall)')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); save(fig, 'supp_recovery_per_class'); plt.show()

## 8 · Answers to the research questions (auto-generated from the numbers)

In [ ]:
B0m = R[R['share'] == 0].groupby('model', observed=True)
lines = []
drop = (B0m['original_attacks_recall'].mean() - B0m['heldout_llm_recall'].mean()).reindex(models_present)
lines.append('RQ1 (recall drop on LLM variants, class-balanced): ' +
             ', '.join(f'{m} {drop[m]:+.3f}' for m in models_present))
ev = rec_cls[rec_cls['share'] == 0].groupby('class')['evasion_mean'].mean().reindex(ATTACK_CLASSES)
lines.append('RQ2 (mean evasion rate over models, per attack type): ' + ', '.join(f'{c} {ev[c]:.3f}' for c in ATTACK_CLASSES))
most = drop.idxmin(); least = drop.idxmax()
lines.append(f'RQ3 (robustness ranking by smallest drop): most robust = {most}, least robust = {least}')
if 'scripted_recall' in R:
    d3 = (B0m['original_attacks_recall'].mean() - B0m['scripted_recall'].mean()).reindex(models_present)
    lines.append('E3 control (drop on scripted noise): ' + ', '.join(f'{m} {d3[m]:+.3f}' for m in models_present) +
                 f' | LLM drop minus noise drop (mean over models): {float((drop - d3).mean()):+.3f}')
for sh in SHARES:
    a = R[R['share'] == sh].groupby('model', observed=True)
    rec = a['recovery'].mean().reindex(models_present); df1 = a['delta_test_f1_macro'].mean().reindex(models_present)
    dfp = a['delta_test_fpr'].mean().reindex(models_present)
    lines.append(f'RQ4 @ {int(sh * 100)} %: recovery ' + ', '.join(f'{m} {rec[m]:+.3f}' for m in models_present) +
                 ' | ΔF1(orig) ' + ', '.join(f'{m} {df1[m]:+.4f}' for m in models_present) +
                 ' | ΔFPR(orig) ' + ', '.join(f'{m} {dfp[m]:+.4f}' for m in models_present))
text = '\n'.join(lines)
print(text)
with open(os.path.join(DIRS['results'], 'FINAL_research_questions.txt'), 'w') as fh:
    fh.write(text + '\n\nNote: values are means over seeds ' + ', '.join(map(str, MODEL_SEEDS)) +
             '; see FINAL_summary_table.csv for standard deviations.\n')

## 9 · Summary of outputs

In [ ]:
print('=' * 72); print('STAGE 5 / EXPERIMENT E4 COMPLETE'); print('=' * 72)
for d, pref in [('results', ('05_', 'FINAL')), ('figures', ('fig', 'supp'))]:
    for f in sorted(os.listdir(DIRS[d])):
        if f.startswith(pref):
            print(f'  {d}/{f}')
print('\nRetrained models:')
for sh in SHARES:
    md_ = os.path.join(DIRS['models'], f'retrained_{int(sh * 100)}pct')
    print(f'  models/retrained_{int(sh * 100)}pct/: {len(os.listdir(md_))} files')

### Notes for the report
- **Recovery** is paired: for each model and seed, held-out recall after retraining minus held-out recall of the *same-seed* baseline. Its std therefore reflects seed-to-seed variation of the improvement itself.
- **Cost of the defence** = ΔF1-macro and ΔFPR on the original, realistic test set (negative ΔF1 / positive ΔFPR = degradation or extra false alarms).
- The held-out variants were never used for training or early stopping. When `source_row_id` was supplied, no real source flow contributes variants to both the pool and the held-out set.
- If E3 shows a drop similar to E2, the effect comes from distribution shift in general rather than from anything specific to the LLM, which is a valid finding (Section 15).